In [1]:
!pip install -q tqdm pandas

import google.genai as genai
from google.genai import types
from google.colab import userdata
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
import pandas as pd
import random
import numpy as np
import re
import asyncio # <--- Added for parallelism
from tqdm.asyncio import tqdm # <--- Async version of tqdm
from collections import Counter
import os

# --- 1. Async Back-Translation Function ---
# Uses 'client.aio' and a semaphore to control speed

async def back_translate_async(text, client, semaphore, target_language="German", temperature=1.5, model="gemini-2.5-flash"):
    """
    Performs back-translation asynchronously.
    Uses a semaphore to limit how many of these run at the exact same time.
    """
    async with semaphore: # Waists here if too many requests are active
        try:
            generation_config = types.GenerateContentConfig(
                temperature=temperature
            )

            # 1. English -> Target
            prompt_to_lang = f"Translate the following text from English to {target_language}. Output the translated text only:\n{text}"

            # Note the use of 'await' and 'client.aio'
            response_lang = await client.aio.models.generate_content(
                model=model,
                contents=[prompt_to_lang],
                config=generation_config
            )
            lang_text = response_lang.text

            # Small delay to be polite to the API even inside the semaphore
            await asyncio.sleep(0.5)

            # 2. Target -> English
            prompt_to_eng = f"Translate the following text from {target_language} to English. Output the translated text only:\n{lang_text}"

            response_eng = await client.aio.models.generate_content(
                model=model,
                contents=[prompt_to_eng],
                config=generation_config
            )
            back_translated_text = response_eng.text

            # 3. Clean and check
            back_translated_text = re.sub(r'\s+', ' ', back_translated_text).strip()

            if not back_translated_text:
                return None

            return back_translated_text

        except Exception as e:
            # If we hit a rate limit (429), print a warning
            if "429" in str(e):
                print(f"--- Rate Limit Hit! Reducing speed recommended. Error: {e}")
            else:
                print(f"--- API call FAILED: {e}")
            return None

# --- 2. Async Dataset Creation Manager ---

async def create_augmented_dataset_async(
    original_dataset,
    csv_path,
    client,
    output_dir,
    target_label,
    target_language,
    concurrency_limit=50, # <--- NEW: Controls how many requests run at once
    temperature=1.5,
    seed=42
):
    random.seed(seed)
    np.random.seed(seed)

    # 1. Prepare Tracking
    def add_tracking_col(example):
        example['is_augmented'] = False
        return example

    print(f"\nPreparing base dataset...")
    base_train_ds = original_dataset['train'].map(add_tracking_col)
    base_test_ds = original_dataset['test'].map(add_tracking_col)

    # 2. Read CSV & Filter Indices
    print(f"Reading CSV file: {csv_path}")
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        raise ValueError(f"Could not read CSV file. Error: {e}")

    if not all(col in df.columns for col in ['index', 'true_label']):
        raise ValueError("CSV must contain columns: 'index' and 'true_label'")

    print(f"Filtering CSV for rows where true_label == '{target_label}'...")
    target_df = df[df['true_label'] == target_label]
    target_indices = set(target_df['index'].tolist())

    print(f"Found {len(target_indices)} rows in CSV matching label '{target_label}'.")
    if len(target_indices) == 0: return

    # 3. Filter Dataset
    rows_to_augment = original_dataset['train'].filter(lambda x: x['index'] in target_indices)
    num_to_augment = len(rows_to_augment)
    print(f"Found {num_to_augment} matching rows in the QEvasion dataset to augment.")
    if num_to_augment == 0: return

    # 4. Prepare Async Tasks
    print(f"Starting PARALLEL Back-Translation (Limit: {concurrency_limit} concurrent requests)...")

    # Create the semaphore "bouncer"
    semaphore = asyncio.Semaphore(concurrency_limit)
    tasks = []

    # Create a task for every row
    for i in range(num_to_augment):
        original_row = rows_to_augment[i]
        text_to_augment = original_row['interview_answer']

        # We pass the original row to the task so we can reconstruct it later
        task = back_translate_async(
            text_to_augment,
            client,
            semaphore,
            target_language=target_language,
            temperature=temperature
        )
        tasks.append(task)

    # 5. Run all tasks and wait for them to finish
    # tqdm.gather displays a progress bar for async tasks
    results = await tqdm.gather(*tasks)

    # 6. Process Results
    new_rows_list = []
    successful_augmentations = 0

    for i, result_text in enumerate(results):
        if result_text:
            new_row = rows_to_augment[i].copy()
            new_row['interview_answer'] = result_text
            new_row['is_augmented'] = True
            new_rows_list.append(new_row)
            successful_augmentations += 1

    print(f"Successfully generated {successful_augmentations} new rows.")

    # 7. Save
    if not new_rows_list:
        new_rows_ds = None
    else:
        new_rows_ds = Dataset.from_list(new_rows_list, features=base_train_ds.features)

    if new_rows_ds:
        final_train_ds = concatenate_datasets([base_train_ds, new_rows_ds])
    else:
        final_train_ds = base_train_ds

    final_dataset = DatasetDict({
        'train': final_train_ds,
        'test': base_test_ds
    })

    print(f"Saving final dataset to {output_dir}...")
    final_dataset.save_to_disk(output_dir)
    print("\n--- Process Complete! ---")

# --- 3. Main Execution Block ---

# Initialize Client
try:
    api_key = userdata.get('GEMINI_API_KEY')
    if not api_key: raise ValueError("GEMINI_API_KEY missing.")
    client = genai.Client(api_key=api_key)
except Exception as e:
    print(f"Error: {e}")
    raise

# Load Data
original_dataset = load_dataset("ailsntua/QEvasion")

# ==========================================
# USER CONFIGURATION
# ==========================================

CSV_FILE_PATH = "combined_output.csv"
TARGET_LABEL = "Clear Reply"
TARGET_LANGUAGE = "German"
TEMPERATURE = 1.5

# **CRITICAL SETTING**: Controls speed.
# Free Tier: Set to 5 or 10.
# Paid Tier: Set to 30 or 50.
CONCURRENCY_LIMIT = 100

OUTPUT_DIR = f"./augmented_{TARGET_LANGUAGE.lower()}_{TARGET_LABEL.replace(' ', '_').lower()}"

# ==========================================

# Run the async function
if os.path.exists(CSV_FILE_PATH):
    # In Colab/Jupyter, we use 'await' directly at the top level
    await create_augmented_dataset_async(
        original_dataset=original_dataset,
        csv_path=CSV_FILE_PATH,
        client=client,
        output_dir=OUTPUT_DIR,
        target_label=TARGET_LABEL,
        target_language=TARGET_LANGUAGE,
        temperature=TEMPERATURE,
        concurrency_limit=CONCURRENCY_LIMIT
    )
else:
    print(f"ERROR: {CSV_FILE_PATH} not found.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.90M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3448 [00:00<?, ? examples/s]

...

100%|██████████| 527/527 [02:39<00:00,  3.31it/s]

Successfully generated 527 new rows.
Saving final dataset to ./augmented_german_clear_reply...


Saving the dataset (0/1 shards):   0%|          | 0/3975 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/308 [00:00<?, ? examples/s]


--- Process Complete! ---


In [2]:
!zip -r augmented_german_clear_reply.zip /content/augmented_german_clear_reply

  adding: content/augmented_german_clear_reply/ (stored 0%)
  adding: content/augmented_german_clear_reply/train/ (stored 0%)
  adding: content/augmented_german_clear_reply/train/data-00000-of-00001.arrow (deflated 82%)
  adding: content/augmented_german_clear_reply/train/dataset_info.json (deflated 84%)
  adding: content/augmented_german_clear_reply/train/state.json (deflated 55%)
  adding: content/augmented_german_clear_reply/test/ (stored 0%)
  adding: content/augmented_german_clear_reply/test/data-00000-of-00001.arrow (deflated 70%)
  adding: content/augmented_german_clear_reply/test/dataset_info.json (deflated 78%)
  adding: content/augmented_german_clear_reply/test/state.json (deflated 38%)
  adding: content/augmented_german_clear_reply/dataset_dict.json (stored 0%)
